In [9]:
# 1 Import Libraries

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier

# 2 Load Dataset

df = pd.read_csv("HR-Employee-Attrition.csv")

# 3 Convert Attrition to numbers

df['Attrition'] = df['Attrition'].map({'Yes':1,'No':0})

# 4 Drop unnecessary columns

df = df.drop(['EmployeeCount','Over18','StandardHours'], axis=1)

# 5 Encode categorical columns

label_encoder = LabelEncoder()

for col in df.columns:
    if df[col].dtype == 'object':
        df[col] = label_encoder.fit_transform(df[col])

# 6 Split features and target

X = df.drop(['Attrition'], axis=1)
y = df['Attrition']

# 7 Train test split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

# 8 Train Model

model = RandomForestClassifier(n_estimators=200, random_state=42)

model.fit(X_train, y_train)

print("Model Training Completed")


# --------------------------------------------------
# FUNCTION TO CHECK EMPLOYEE
# --------------------------------------------------

def check_employee(employee_number):

    emp = X[X['EmployeeNumber'] == employee_number]

    if emp.empty:
        print("Employee not found in dataset")
        return

    prediction = model.predict(emp)
    probability = model.predict_proba(emp)

    risk = probability[0][1] * 100

    print("\nAttrition Risk Score:", round(risk,2), "%")

    if prediction[0] == 1:

        print("\nPrediction: Employee likely to LEAVE the company")

        print("\nReasons & HR Suggestions:")

        overtime = df.loc[df['EmployeeNumber']==employee_number,'OverTime'].values[0]
        satisfaction = df.loc[df['EmployeeNumber']==employee_number,'JobSatisfaction'].values[0]
        worklife = df.loc[df['EmployeeNumber']==employee_number,'WorkLifeBalance'].values[0]
        income = df.loc[df['EmployeeNumber']==employee_number,'MonthlyIncome'].values[0]

        if overtime == 1:
            print("- High overtime → Reduce workload")

        if satisfaction <= 2:
            print("- Low job satisfaction → Improve recognition")

        if worklife <= 2:
            print("- Poor work life balance → Flexible schedule")

        if income < 4000:
            print("- Low salary → Consider salary increment")

    else:

        print("\nPrediction: Employee likely to STAY in the company")

        print("\nHR Suggestion:")
        print("- Maintain engagement and growth opportunities")


# --------------------------------------------------
# TOP 5 EMPLOYEES LIKELY TO LEAVE
# --------------------------------------------------

def top_risk_employees():

    probabilities = model.predict_proba(X)

    risk_scores = probabilities[:,1]

    df['RiskScore'] = risk_scores

    top5 = df.sort_values(by='RiskScore', ascending=False).head(5)

    print("\nTop 5 Employees Likely to Leave:\n")

    for index, row in top5.iterrows():

        print("Employee ID:", row['EmployeeNumber'])
        print("Risk Score:", round(row['RiskScore']*100,2), "%")

        print("Possible Reasons:")

        if row['OverTime'] == 1:
            print("- High overtime")

        if row['JobSatisfaction'] <= 2:
            print("- Low job satisfaction")

        if row['WorkLifeBalance'] <= 2:
            print("- Poor work life balance")

        if row['MonthlyIncome'] < 4000:
            print("- Low salary")

        print("---------------------------------")


# --------------------------------------------------
# USER INPUT
# --------------------------------------------------

emp_id = int(input("\nEnter Employee Number: "))

check_employee(emp_id)

print("\n---------------------------------")

top_risk_employees()

from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print("Model Accuracy:", round(accuracy*100,2), "%")


Model Training Completed



Enter Employee Number:  878



Attrition Risk Score: 7.5 %

Prediction: Employee likely to STAY in the company

HR Suggestion:
- Maintain engagement and growth opportunities

---------------------------------

Top 5 Employees Likely to Leave:

Employee ID: 622.0
Risk Score: 95.5 %
Possible Reasons:
- High overtime
- Poor work life balance
- Low salary
---------------------------------
Employee ID: 1868.0
Risk Score: 93.0 %
Possible Reasons:
- High overtime
- Poor work life balance
- Low salary
---------------------------------
Employee ID: 959.0
Risk Score: 92.5 %
Possible Reasons:
- High overtime
- Low job satisfaction
- Low salary
---------------------------------
Employee ID: 1273.0
Risk Score: 92.0 %
Possible Reasons:
- High overtime
- Low salary
---------------------------------
Employee ID: 514.0
Risk Score: 90.5 %
Possible Reasons:
- High overtime
- Low salary
---------------------------------
Model Accuracy: 87.41 %
